In [ ]:
import pvlib

import os
os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "full"
#os.environ["HOSTRADA_NETCDF_SUBSET_MODE"] = "auto"

## Selecting a location using the OSM map and address search

Select the location for generating weather data by clicking on the map, dragging the marker, entering the longitude and latitude directly, or by entering an address. The variables `selected_lon` and `selected_lat` are automatically updated and used in the subsequent export cells. The center of the map is Berlin-Alexanderplatz.


In [ ]:
ALEXANDERPLATZ_LAT = 52.521918
ALEXANDERPLATZ_LON = 13.413215

selected_lat = ALEXANDERPLATZ_LAT
selected_lon = ALEXANDERPLATZ_LON
selected_address = "Berlin Alexanderplatz"

try:
    from IPython.display import display
    from ipyleaflet import Map, Marker, basemaps, LayersControl
    from ipywidgets import FloatText, HBox, VBox, HTML, Text, Button, Layout
    import json
    import urllib.parse
    import urllib.request
    import urllib.error

    lat_widget = FloatText(
        value=selected_lat,
        description="Latitude",
        step=0.000001,
        layout=Layout(width="260px"),
    )
    lon_widget = FloatText(
        value=selected_lon,
        description="Longitude",
        step=0.000001,
        layout=Layout(width="260px"),
    )
    address_widget = Text(
        value=selected_address,
        placeholder="Enter address, e.g. Hardenbergstraße 33, Berlin",
        description="Address",
        layout=Layout(width="620px"),
    )
    search_button = Button(
        description="Search Address",
        button_style="primary",
        tooltip="Search for an address using OpenStreetMap",
        icon="search",
        layout=Layout(width="160px"),
    )
    status_widget = HTML(value="")

    weather_location_marker = Marker(
        location=(selected_lat, selected_lon),
        draggable=True,
        title="Selected Location",
    )

    weather_location_map = Map(
        center=(ALEXANDERPLATZ_LAT, ALEXANDERPLATZ_LON),
        zoom=13,
        basemap=basemaps.OpenStreetMap.Mapnik,
        scroll_wheel_zoom=True,
        layout={"height": "520px", "width": "100%"},
    )
    weather_location_map.add(weather_location_marker)
    weather_location_map.add(LayersControl(position="topright"))

    _updating_widgets = False

    def _set_selected_location(lat, lon, update_marker=True, center_map=False, zoom=None, address=None):
        """Synchronizes markers, widgets, and the global variables selected_lat and selected_lon."""
        global selected_lat, selected_lon, selected_address, _updating_widgets
        selected_lat = float(lat)
        selected_lon = float(lon)
        if address is not None:
            selected_address = str(address)

        _updating_widgets = True
        lat_widget.value = selected_lat
        lon_widget.value = selected_lon
        if address is not None:
            address_widget.value = selected_address
        _updating_widgets = False

        if update_marker:
            weather_location_marker.location = (selected_lat, selected_lon)
        if center_map:
            weather_location_map.center = (selected_lat, selected_lon)
            if zoom is not None:
                weather_location_map.zoom = zoom

    def _on_marker_moved(change):
        lat, lon = change["new"]
        _set_selected_location(lat, lon, update_marker=False)
        status_widget.value = (
            f"Gewählter Standort: lat={selected_lat:.6f}, lon={selected_lon:.6f}"
        )

    def _on_map_clicked(**kwargs):
        if kwargs.get("type") == "click":
            lat, lon = kwargs.get("coordinates")
            _set_selected_location(lat, lon, update_marker=True)
            status_widget.value = (
                f"Gewählter Standort: lat={selected_lat:.6f}, lon={selected_lon:.6f}"
            )

    def _on_widget_changed(change):
        if not _updating_widgets:
            _set_selected_location(lat_widget.value, lon_widget.value, update_marker=True, center_map=True)
            status_widget.value = (
                f"Gewählter Standort: lat={selected_lat:.6f}, lon={selected_lon:.6f}"
            )

    def _geocode_address(address, *, timeout=15):
        """Searches for an address using OSM/Nominatim and returns (lat, lon, display_name)."""
        query = address.strip()
        if not query:
            raise ValueError("Please, enter a address.")

        params = urllib.parse.urlencode({
            "q": query,
            "format": "json",
            "limit": 1,
            "addressdetails": 1,
        })
        url = f"https://nominatim.openstreetmap.org/search?{params}"
        request = urllib.request.Request(
            url,
            headers={
                # Nominatim verlangt einen identifizierbaren User-Agent.
                "User-Agent": "hostrada4py-weather-notebook/1.0 (Jupyter Notebook)",
                "Accept": "application/json",
            },
        )
        with urllib.request.urlopen(request, timeout=timeout) as response:
            payload = json.loads(response.read().decode("utf-8"))

        if not payload:
            raise LookupError(f"Keine Koordinaten für diese Adresse gefunden: {query}")

        hit = payload[0]
        return float(hit["lat"]), float(hit["lon"]), hit.get("display_name", query)

    def search_address(_=None):
        """Perform an address search and place a marker on the map at the location."""
        search_button.disabled = True
        status_widget.value = "Search for an address using OpenStreetMap/Nominatim ..."
        try:
            lat, lon, label = _geocode_address(address_widget.value)
            _set_selected_location(lat, lon, update_marker=True, center_map=True, zoom=15, address=label)
            status_widget.value = (
                "Found: "
                f"{label}<br>lat={selected_lat:.6f}, lon={selected_lon:.6f}"
            )
        except Exception as exc:
            status_widget.value = (
                "<b>Address search failed:</b> "
                f"{type(exc).__name__}: {exc}<br>"
                "You can still select the location using a map or by entering coordinates."
            )
        finally:
            search_button.disabled = False

    weather_location_marker.observe(_on_marker_moved, names="location")
    weather_location_map.on_interaction(_on_map_clicked)
    lat_widget.observe(_on_widget_changed, names="value")
    lon_widget.observe(_on_widget_changed, names="value")
    search_button.on_click(search_address)
    address_widget.on_submit(search_address)

    display(VBox([
        HTML("<b>Select Location:</b> Click on the map, move the marker, or search for an address."),
        HBox([address_widget, search_button]),
        status_widget,
        weather_location_map,
        HBox([lat_widget, lon_widget]),
        HTML("The following cells use <code>selected_lon</code> and <code>selected_lat</code>."),
    ]))

except ImportError as exc:
    from IPython.display import display, Markdown
    display(Markdown(
        "**Hint:** The interactive map requires `ipyleaflet` and `ipywidgets`. "
        "Install these packages using, for example, `pip install ipyleaflet ipywidgets`.\n\n"
        "Until then, the coordinates for Berlin-Alexanderplatz will be used as the default."
    ))
    print(f"Map widget not loaded: {exc}")


## Time period for the weather file generation

Select the start and end date with the mouse using the date picker widgets. The variables `selected_start` and `selected_end` are automatically updated and used in the subsequent export cells.


In [ ]:
from datetime import date, datetime

# Default period: full year 2025
selected_start = "2025-01-01 00:00"
selected_end = "2025-12-31 23:00"
selected_start_datetime = datetime(2025, 1, 1, 0, 0)
selected_end_datetime = datetime(2025, 12, 31, 23, 0)

try:
    from IPython.display import display
    from ipywidgets import DatePicker, Dropdown, HBox, VBox, HTML, Layout

    start_date_widget = DatePicker(
        description="Start date",
        value=date(2025, 1, 1),
        layout=Layout(width="260px"),
    )
    start_hour_widget = Dropdown(
        description="Start hour",
        options=[(f"{hour:02d}:00", hour) for hour in range(24)],
        value=0,
        layout=Layout(width="220px"),
    )
    end_date_widget = DatePicker(
        description="End date",
        value=date(2025, 12, 31),
        layout=Layout(width="260px"),
    )
    end_hour_widget = Dropdown(
        description="End hour",
        options=[(f"{hour:02d}:00", hour) for hour in range(24)],
        value=23,
        layout=Layout(width="220px"),
    )
    period_status_widget = HTML(value="")

    def _update_selected_period(*_):
        """Synchronizes date/hour widgets and the global weather-period variables."""
        global selected_start, selected_end, selected_start_datetime, selected_end_datetime

        if start_date_widget.value is None or end_date_widget.value is None:
            period_status_widget.value = "<b>Bitte Start- und Enddatum auswählen.</b>"
            return

        selected_start_datetime = datetime.combine(
            start_date_widget.value,
            datetime.min.time(),
        ).replace(hour=int(start_hour_widget.value), minute=0, second=0, microsecond=0)

        selected_end_datetime = datetime.combine(
            end_date_widget.value,
            datetime.min.time(),
        ).replace(hour=int(end_hour_widget.value), minute=0, second=0, microsecond=0)

        selected_start = selected_start_datetime.strftime("%Y-%m-%d %H:%M")
        selected_end = selected_end_datetime.strftime("%Y-%m-%d %H:%M")

        if selected_start_datetime > selected_end_datetime:
            period_status_widget.value = (
                "<b style='color:#b00020'>Unvalid time period:</b> "
                f"Start {selected_start} liegt nach Ende {selected_end}."
            )
        else:
            period_status_widget.value = (
                f"Gewählter Zeitraum: <b>{selected_start}</b> bis <b>{selected_end}</b>"
            )

    for widget in [start_date_widget, start_hour_widget, end_date_widget, end_hour_widget]:
        widget.observe(_update_selected_period, names="value")

    _update_selected_period()

    display(VBox([
        HTML("<b>Select time period:</b> Choose start and end date/hour for the generated weather file."),
        HBox([start_date_widget, start_hour_widget]),
        HBox([end_date_widget, end_hour_widget]),
        period_status_widget,
        HTML("The following cells use <code>selected_start</code> and <code>selected_end</code>."),
    ]))

except ImportError as exc:
    from IPython.display import display, Markdown
    display(Markdown(
        "**Hint:** The interactive time-period selection requires `ipywidgets`. "
        "Install it using, for example, `pip install ipywidgets`.\n\n"
        "Until then, the default period `2025-01-01 00:00` to `2025-12-31 23:00` will be used."
    ))
    print(f"Time-period widget not loaded: {exc}")


## Generation of IDA ICE weather Files

In [ ]:
from hostrada4py import hostrada_IDA_ICE_Weather as iw
iw.create_ida_ice_weather_file(
    # Standort aus der OSM-Karte oben:
    lon = selected_lon,
    lat = selected_lat,
    # Zeitraum aus der Eingabemaske oben:
    start=selected_start,
    end=selected_end,
    output_file="HOSTRADA_IDA_ICE.prn",
    tz="Europe/Berlin",
)


## Generation of Polysun weather files

In [ ]:
from hostrada4py import hostrada_Polysun_Weather as pw
pw.create_polysun_weather_file(
    # Standort aus der OSM-Karte oben:
    lon = selected_lon,
    lat = selected_lat,
    # Zeitraum aus der Eingabemaske oben:
    start=selected_start,
    end=selected_end,
    output_file="HOSTRADA_Polysun.csv",
    tz="Europe/Berlin",
)


## Generation of EnergyPlus weather files

In [ ]:
from hostrada4py import hostrada_EnergyPlus_Weather as epw
epw.create_energyplus_weather_file(
    # Standort aus der OSM-Karte oben:
    lon = selected_lon,
    lat = selected_lat,
    # Zeitraum aus der Eingabemaske oben:
    start=selected_start,
    end=selected_end,
    output_file="HOSTRADA_EnergyPlus.epw",
    tz="Europe/Berlin",
)


## Generation of SimStadt weather files


In [ ]:
from hostrada4py import hostrada_SimStadt_Weather as ssw
ssw.create_simstadt_weather_file(
    # Standort aus der OSM-Karte oben:
    lon = selected_lon,
    lat = selected_lat,
    # Zeitraum aus der Eingabemaske oben:
    start=selected_start,
    end=selected_end,
    output_file="HOSTRADA_SimStadt.tmy3",
    location_name="HOSTRADA_SimStadt",
    altitude=34.0,
    tz="Europe/Berlin",
)


## Generation of BuildingSystems weather files

In [ ]:
from hostrada4py import hostrada_BuildingSystems_Weather as bsw
bsw.create_buildingsystems_csv_weather_file(
    # Standort aus der OSM-Karte oben:
    lon = selected_lon,
    lat = selected_lat,
    # Zeitraum aus der Eingabemaske oben:
    start=selected_start,
    end=selected_end,
    output_file="HOSTRADA_BuildingSystems.csv",
    tz="Europe/Berlin",
)


## Visualization of the generated weather data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, HTML, clear_output

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    _PLOTLY_AVAILABLE = True
except Exception as _plotly_error:
    _PLOTLY_AVAILABLE = False
    _PLOTLY_IMPORT_ERROR = _plotly_error

try:
    import ipywidgets as widgets
    _WIDGETS_AVAILABLE = True
except Exception as _widgets_error:
    _WIDGETS_AVAILABLE = False
    _WIDGETS_IMPORT_ERROR = _widgets_error


def _current_start_timestamp():
    """Return the selected start timestamp used to reconstruct file time axes."""
    for name in ("selected_start_datetime", "selected_start"):
        if name in globals():
            try:
                return pd.Timestamp(globals()[name])
            except Exception:
                pass
    return pd.Timestamp("2025-01-01 00:00")


def _hourly_index(n_rows):
    return pd.date_range(start=_current_start_timestamp(), periods=int(n_rows), freq="h")


def _read_ida_ice_weather(path):
    path = Path(path)
    df = pd.read_csv(path, sep=r"\s+|;|,", engine="python", comment="#")
    temperature_col = "DryBulb_C" if "DryBulb_C" in df.columns else df.columns[3]
    radiation_col = "GlobalHorizontal_W_m2" if "GlobalHorizontal_W_m2" in df.columns else df.columns[-1]
    return pd.DataFrame({
        "time": _hourly_index(len(df)),
        "temperature_C": pd.to_numeric(df[temperature_col], errors="coerce"),
        "global_radiation": pd.to_numeric(df[radiation_col], errors="coerce"),
    }), "W/m²"


def _read_polysun_weather(path):
    path = Path(path)
    df = pd.read_csv(path, comment="#")
    temperature_col = "Tamb" if "Tamb" in df.columns else df.columns[3]
    radiation_col = "Gh" if "Gh" in df.columns else df.columns[0]
    return pd.DataFrame({
        "time": _hourly_index(len(df)),
        "temperature_C": pd.to_numeric(df[temperature_col], errors="coerce"),
        "global_radiation": pd.to_numeric(df[radiation_col], errors="coerce"),
    }), "Wh/m²"


def _read_energyplus_weather(path):
    path = Path(path)
    epw_columns = [
        "Year", "Month", "Day", "Hour", "Minute", "Data Source and Uncertainty Flags",
        "Dry Bulb Temperature", "Dew Point Temperature", "Relative Humidity",
        "Atmospheric Station Pressure", "Extraterrestrial Horizontal Radiation",
        "Extraterrestrial Direct Normal Radiation", "Horizontal Infrared Radiation Intensity",
        "Global Horizontal Radiation", "Direct Normal Radiation", "Diffuse Horizontal Radiation",
        "Global Horizontal Illuminance", "Direct Normal Illuminance", "Diffuse Horizontal Illuminance",
        "Zenith Luminance", "Wind Direction", "Wind Speed", "Total Sky Cover", "Opaque Sky Cover",
        "Visibility", "Ceiling Height", "Present Weather Observation", "Present Weather Codes",
        "Precipitable Water", "Aerosol Optical Depth", "Snow Depth", "Days Since Last Snowfall",
        "Albedo", "Liquid Precipitation Depth", "Liquid Precipitation Quantity",
    ]
    df = pd.read_csv(path, skiprows=8, header=None, names=epw_columns)
    hour_zero_based = pd.to_numeric(df["Hour"], errors="coerce").fillna(1).astype(int) - 1
    hour_zero_based = hour_zero_based.clip(lower=0, upper=23)
    time = pd.to_datetime({
        "year": pd.to_numeric(df["Year"], errors="coerce").astype(int),
        "month": pd.to_numeric(df["Month"], errors="coerce").astype(int),
        "day": pd.to_numeric(df["Day"], errors="coerce").astype(int),
        "hour": hour_zero_based,
    })
    return pd.DataFrame({
        "time": time,
        "temperature_C": pd.to_numeric(df["Dry Bulb Temperature"], errors="coerce"),
        "global_radiation": pd.to_numeric(df["Global Horizontal Radiation"], errors="coerce"),
    }), "W/m²"


def _read_simstadt_tmy3_weather(path):
    path = Path(path)
    df = pd.read_csv(path, skiprows=1)
    temperature_col = "Dry-bulb (C)" if "Dry-bulb (C)" in df.columns else df.columns[31]
    radiation_col = "GHI (W/m^2)" if "GHI (W/m^2)" in df.columns else df.columns[4]
    return pd.DataFrame({
        "time": _hourly_index(len(df)),
        "temperature_C": pd.to_numeric(df[temperature_col], errors="coerce"),
        "global_radiation": pd.to_numeric(df[radiation_col], errors="coerce"),
    }), "W/m²"


def _read_buildingsystems_csv_weather(path):
    path = Path(path)
    columns = [
        "Time_h", "Time_s", "TAirRef_degC", "RelHum_pct",
        "GlobalHorizontalRadiation_W_m2", "DiffuseHorizontalRadiation_W_m2",
        "CloudCover_okta", "WindSpeed_m_s", "WindDirection_deg",
    ]
    df = pd.read_csv(path, comment="#", header=None)
    if len(df) > 0 and str(df.iloc[0, 0]).strip() == "Time_h":
        df = df.iloc[1:].reset_index(drop=True)
    df = df.iloc[:, :len(columns)]
    df.columns = columns[:df.shape[1]]
    if "Time_s" in df.columns:
        seconds = pd.to_numeric(df["Time_s"], errors="coerce")
        time = _current_start_timestamp() + pd.to_timedelta(seconds - seconds.iloc[0], unit="s")
    else:
        time = _hourly_index(len(df))
    return pd.DataFrame({
        "time": time,
        "temperature_C": pd.to_numeric(df["TAirRef_degC"], errors="coerce"),
        "global_radiation": pd.to_numeric(df["GlobalHorizontalRadiation_W_m2"], errors="coerce"),
    }), "W/m²"


WEATHER_FILE_CONFIGS = {
    "ida_ice": {
        "label": "IDA ICE (*.prn)",
        "file_name": "HOSTRADA_IDA_ICE.prn",
        "reader": _read_ida_ice_weather,
        "title": "IDA ICE: Outdoor air temperature and global radiation",
    },
    "polysun": {
        "label": "Polysun (*.csv)",
        "file_name": "HOSTRADA_Polysun.csv",
        "reader": _read_polysun_weather,
        "title": "Polysun: Outdoor air temperature and global radiation",
    },
    "energyplus": {
        "label": "EnergyPlus (*.epw)",
        "file_name": "HOSTRADA_EnergyPlus.epw",
        "reader": _read_energyplus_weather,
        "title": "EnergyPlus: Outdoor air temperature and global radiation",
    },
    "simstadt": {
        "label": "SimStadt (*.tmy3)",
        "file_name": "HOSTRADA_SimStadt.tmy3",
        "reader": _read_simstadt_tmy3_weather,
        "title": "SimStadt: Outdoor air temperature and global radiation",
    },
    "buildingsystems_csv": {
        "label": "BuildingSystems CSV (*.csv)",
        "file_name": "HOSTRADA_BuildingSystems.csv",
        "reader": _read_buildingsystems_csv_weather,
        "title": "BuildingSystems CSV: Outdoor air temperature and global radiationg",
    },
}


def available_generated_weather_files():
    """Return a dict of generated weather files that currently exist in the notebook folder."""
    return {
        key: cfg for key, cfg in WEATHER_FILE_CONFIGS.items()
        if Path(cfg["file_name"]).exists()
    }


def create_weather_timeseries_figure(file_key):
    """Create a zoomable Plotly figure for one generated weather file."""
    if not _PLOTLY_AVAILABLE:
        raise ImportError(f"Plotly ist nicht verfügbar: {_PLOTLY_IMPORT_ERROR}")
    if file_key not in WEATHER_FILE_CONFIGS:
        raise KeyError(f"Unbekannter Wetterdatei-Typ: {file_key}")
    cfg = WEATHER_FILE_CONFIGS[file_key]
    path = Path(cfg["file_name"])
    if not path.exists():
        raise FileNotFoundError(
            f"{cfg['file_name']} wurde nicht gefunden. Bitte zuerst die passende Generierungszelle ausführen."
        )
    data, radiation_unit = cfg["reader"](path)
    data = data.dropna(subset=["time"])
    if data.empty:
        raise ValueError(f"Keine verwertbaren Zeitreihendaten in {cfg['file_name']} gefunden.")

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=data["time"],
            y=data["temperature_C"],
            name="Outdoor air temperature [°C]",
            mode="lines",
            hovertemplate="%{x}<br>Outdoor air temperature: %{y:.2f} °C<extra></extra>",
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=data["time"],
            y=data["global_radiation"],
            name=f"Global radiation [{radiation_unit}]",
            mode="lines",
            hovertemplate=f"%{{x}}<br>Global radiation: %{{y:.2f}} {radiation_unit}<extra></extra>",
        ),
        secondary_y=True,
    )
    fig.update_layout(
        title=cfg["title"],
        xaxis_title="Zeit",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        dragmode="zoom",
        height=560,
        margin=dict(l=70, r=90, t=95, b=75),
    )
    fig.update_xaxes(
        rangeslider=dict(visible=True),
        rangeselector=dict(buttons=[
            dict(count=1, label="1d", step="day", stepmode="backward"),
            dict(count=7, label="7d", step="day", stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(step="all", label="alles"),
        ]),
    )
    fig.update_yaxes(title_text="Outdoor air temperature [°C]", secondary_y=False)
    fig.update_yaxes(title_text=f"Global radiation [{radiation_unit}]", secondary_y=True)
    return fig


def display_plotly_html(fig):
    """Display Plotly as explicit HTML instead of relying on the active notebook renderer."""
    html = pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs="cdn",
        config={
            "scrollZoom": True,
            "responsive": True,
            "displaylogo": False,
            "modeBarButtonsToAdd": ["drawline", "eraseshape"],
        },
    )
    display(HTML(html))


def plot_weather_file(file_key):
    """Display one weather file by key: ida_ice, polysun, energyplus, simstadt, buildingsystems_csv."""
    fig = create_weather_timeseries_figure(file_key)
    display_plotly_html(fig)
    return fig


def plot_all_weather_files():
    """Display all generated weather files that exist in the notebook folder."""
    existing = available_generated_weather_files()
    if not existing:
        display(HTML(
            "<b>No weather files found yet.</b><br>"
            "Please run at least one generation cell first. "
            "Then restart this visualization cell."
        ))
        return []
    figures = []
    for file_key in existing:
        try:
            display(HTML(f"<h4>{WEATHER_FILE_CONFIGS[file_key]['label']}</h4>"))
            figures.append(plot_weather_file(file_key))
        except Exception as exc:
            display(HTML(f"<b>{WEATHER_FILE_CONFIGS[file_key]['file_name']}:</b> Visualization not possible: {exc}"))
    return figures


def show_weather_file_status():
    rows = []
    for key, cfg in WEATHER_FILE_CONFIGS.items():
        path = Path(cfg["file_name"])
        status = "available" if path.exists() else "not found"
        size = f"{path.stat().st_size / 1024:.1f} kB" if path.exists() else "-"
        rows.append(f"<tr><td><code>{key}</code></td><td>{cfg['file_name']}</td><td>{status}</td><td>{size}</td></tr>")
    display(HTML(
        "<b>Status of the weather files in the current notebook directory</b>"
        "<table><tr><th>Schlüssel</th><th>Datei</th><th>Status</th><th>Größe</th></tr>"
        + "".join(rows) + "</table>"
    ))


def display_weather_file_selector():
    """Display selector if ipywidgets is available. Direct function calls below work without widgets."""
    if not _PLOTLY_AVAILABLE:
        display(HTML(
            "<b>Plotly ist not installed.</b><br>"
            "Please run it once: <code>pip install plotly</code>"
        ))
        return None
    show_weather_file_status()
    if not _WIDGETS_AVAILABLE:
        display(HTML(
            "<b>ipywidgets ist nicht installiert.</b><br>"
            "The direct plot calls in the following cells still work, for example:"
            "<code>plot_weather_file('energyplus')</code>."
        ))
        return None

    existing = available_generated_weather_files()
    option_items = [("All available weather files", "__all__")]
    for key, cfg in WEATHER_FILE_CONFIGS.items():
        suffix = " ✓" if key in existing else " (not yet generated)"
        option_items.append((cfg["label"] + suffix, key))

    dropdown = widgets.Dropdown(
        options=option_items,
        value="__all__",
        description="File:",
        layout=widgets.Layout(width="560px"),
    )
    button = widgets.Button(
        description="Show visualization",
        button_style="primary",
        icon="line-chart",
        layout=widgets.Layout(width="240px"),
    )
    output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))

    def _render_selection(_=None):
        with output:
            clear_output(wait=True)
            if dropdown.value == "__all__":
                plot_all_weather_files()
            else:
                try:
                    plot_weather_file(dropdown.value)
                except Exception as exc:
                    cfg = WEATHER_FILE_CONFIGS[dropdown.value]
                    display(HTML(
                        f"<b>{cfg['file_name']}</b> kann noch nicht dargestellt werden.<br>"
                        f"{exc}"
                    ))

    button.on_click(_render_selection)
    ui = widgets.VBox([
        widgets.HTML(
            "<b>Visualization of the generated weather data</b><br>"
            "Select a single file or all available files. "
        ),
        widgets.HBox([dropdown, button]),
        output,
    ])
    globals()["weather_file_selector_ui"] = ui
    display(ui)
    _render_selection()
    return None

_weather_file_selector_result = display_weather_file_selector()
